# FVAPFPEVF in the DPP-IV active site

The best HADDOCK pose of the Tier-1 lead bound to DPP-IV (4A5S), rendered interactively.

Cluster 9 is the best-scoring of the ten binding modes this run produced, at -100.14 mean against -97.76 for the next and the tightest spread of the ten (SD 1.55 against 7.02). Model 1 is its top-ranked member at -102.25. This is the cluster phase5.5_docking.ipynb already reports as the peptide's result, so the -100.1 in Table 10 is cluster 9's mean rather than a separate figure.

Contacts are heavy-atom pairs, counted within 5 A in the exported tables and drawn at 4.5 A in the view.

    INPUTS  docking/haddock_runs/734078-DPP4_FVAPFPEVF_summary.tgz  (the run archive)

    OUTPUTS  results/phase5_5_pose_contacts.csv  - per-residue nearest site residue and contact count
             results/phase5_5_pose_contacts_full.csv  - every contact within 5 A, with atom names

    REQUIREMENTS  pip install py3Dmol

In [ ]:
# Imports
import csv
import math
import os
import tarfile

import py3Dmol

# same as before, due to moving file to folder late on in the process
while not os.path.isdir("docking") and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir("..")

In [2]:
# cluster 9 of the ten this run produced, model 1 of its four. the markdown above carries
# the scores. hard-coded rather than recomputed, so it has to stay in step with the best_cl
# that phase5.5_docking.ipynb writes into phase5_5_docking.csv
POSE = "cluster9_1.pdb"

# the job number in the filename is what ties this pose to one portal run
ARCHIVE_DIR = os.environ.get("HADDOCK_DIR", "docking/haddock_runs")
TGZ = os.path.join(ARCHIVE_DIR, "734078-DPP4_FVAPFPEVF_summary.tgz")

with tarfile.open(TGZ) as tar:
    member = next((m for m in tar.getmembers() if m.name.endswith(POSE)), None)
    assert member is not None, f"{POSE} not found in {TGZ}"
    pdb = tar.extractfile(member).read().decode()

In [3]:
# residue maps. 4A5S numbering is preserved through docking and chain B is FVAPFPEVF
PEP  = {1:"Phe1", 2:"Val2", 3:"Ala3", 4:"Pro4", 5:"Phe5", 6:"Pro6", 7:"Glu7", 8:"Val8", 9:"Phe9"}

# the eight site residues the figure draws, not the full 22. seventeen pairs fall inside
# CUTOFF across the whole site, more dashed lines than one view can carry, so this is the
# legible subset. cell 4 exports every contact over all 22
SITE = {125: "Arg125", 205: "Glu205", 206: "Glu206", 545: "Asp545",
        630: "Ser630", 662: "Tyr662", 666: "Tyr666", 740: "His740"}

CUTOFF = 4.5 # heavy-atom contact. at 4.0 the Val8-His740 contact is missed by 0.02 A
CATALYTIC = {630, 740} # two of the Ser630/Asp708/His740 triad


def heavy(chain, resi):
    """Heavy-atom coordinates of one residue, hydrogens dropped."""
    # every hydrogen in this model is named starting with H, so the name test is enough
    out = []
    for ln in pdb.splitlines():
        if ln.startswith(("ATOM", "HETATM")) and ln[21] == chain and int(ln[22:26]) == resi:
            if not ln[12:16].strip().startswith("H"):
                out.append((float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
    return out


def closest(a, b):
    """Shortest distance between two coordinate lists, as (distance, point_a, point_b)."""
    best = None
    for p in a:
        for q in b:
            d = math.dist(p, q)
            if best is None or d < best[0]:
                best = (d, p, q)
    return best


# every peptide residue against every drawn site residue. anything inside CUTOFF becomes a
# dashed line carrying its distance
contacts = []
for pr in PEP:
    pa = heavy("B", pr)
    for sr in SITE:
        c = closest(pa, heavy("A", sr))
        if c and c[0] <= CUTOFF:
            contacts.append((pr, sr, c[0], c[1], c[2]))

for pr, sr, d, _, _ in contacts:
    print(f"{PEP[pr]} - {SITE[sr]}: {d:.1f} A")

# Phe1 and Pro4 are labelled whether or not they contact
LABEL_PEP  = sorted({1, 4} | {pr for pr, _, _, _, _ in contacts})
SITE_SHOWN = sorted({sr for _, sr, _, _, _ in contacts}) # contacted site residues, drawn

Phe1 - Glu205: 3.3 A
Phe1 - Glu206: 3.3 A
Pro4 - Arg125: 3.4 A
Phe5 - Ser630: 3.5 A
Val8 - His740: 4.0 A
Phe9 - Asp545: 2.9 A


In [4]:
# the same 22 restrained residues the docking used, read from the file that run wrote
# rather than restated, as phase5.7 does
line = next(l for l in open("docking/haddock_active_residues.txt") if l.startswith("receptor"))
SITE_ALL = [int(n) for n in line.split(":", 1)[1].split(",")]


def atoms(chain, resi):
    """Heavy atoms of one residue as (atom_name, xyz)."""
    return [(ln[12:16].strip(), (float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
            for ln in pdb.splitlines()
            if ln.startswith(("ATOM", "HETATM")) and ln[21] == chain
            and int(ln[22:26]) == resi and not ln[12:16].strip().startswith("H")]


def resname(chain, resi):
    """Three-letter name of one receptor residue."""
    for ln in pdb.splitlines():
        if ln.startswith("ATOM") and ln[21] == chain and int(ln[22:26]) == resi:
            return ln[17:20]
    return "?"


def nearest_atoms(a, b):
    """Shortest distance between two atom lists, with both atom names."""
    return min((math.dist(p, q), na, nb) for na, p in a for nb, q in b)


# per-residue summary. what each position sits nearest to, its distance to the two catalytic
# residues, and how many of the 22 it reaches within 5 A
rows = []
for pr in PEP:
    pa = heavy("B", pr)
    per = {sr: closest(pa, heavy("A", sr)) for sr in SITE_ALL}
    per = {sr: c[0] for sr, c in per.items() if c}
    near = min(per, key=per.get)
    rows.append({"residue": PEP[pr], "position": pr,
                 "nearest_site_residue": near, "nearest_dist": round(per[near], 2),
                 "dist_Ser630": round(per[630], 2), "dist_His740": round(per[740], 2),
                 "n_contacts_within_5A": sum(d <= 5.0 for d in per.values())})

os.makedirs("results", exist_ok=True)
with open("results/phase5_5_pose_contacts.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0]))
    w.writeheader()
    w.writerows(rows)

# every contact within 5 A with both atom names, since section 3.5 looks at which atom makes
# each contact rather than on the distance alone
with open("results/phase5_5_pose_contacts_full.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["pep_residue", "pep_atom", "site_residue", "site_atom", "distance"])
    for pr in PEP:
        pa = atoms("B", pr)
        for sr in SITE_ALL:
            sb = atoms("A", sr)
            if not sb:
                continue
            d, na, nb = nearest_atoms(pa, sb)
            if d <= 5.0:
                w.writerow([PEP[pr], na, f"{resname('A', sr)}{sr}", nb, round(d, 2)])

    # the free alpha-amino nitrogen on its own, since the N-terminal claim turns on it
    n1 = [(na, p) for na, p in atoms("B", 1) if na == "N"]
    for sr in (205, 206):
        d, na, nb = nearest_atoms(n1, atoms("A", sr))
        w.writerow(["Phe1 alpha-amino N", na, f"{resname('A', sr)}{sr}", nb, round(d, 2)])

In [5]:
# pixel offsets that pull apart distance labels landing on top of each other. tuned
# for this pose and camera, at a nominal 800 px width, then scaled with the canvas
LABEL_NUDGE = {(5, 630): (0, -22), (8, 740): (0, 22),
               (1, 205): (0, -16), (1, 206): (0, 16),
               (9, 545): (30, -15), (4, 125): (0, -18)}


def build_view(zoom=1.4, size=(1400, 1000)):
    """The pose, its contacted site residues and every drawn contact, zoomed by `zoom`."""
    w, h = size
    view = py3Dmol.view(width=w, height=h)
    view.addModel(pdb, "pdb")
    view.setBackgroundColor("white")

    # font sizes, line weights and label offsets are absolute pixels in py3Dmol, so scale
    # them against the same nominal 800 px or they shrink as the canvas grows
    s = w / 800
    FS_SITE = int(16*s)
    FS_PEP = int(16*s)
    FS_DIST = int(14*s)
    LW = 3 * s

    # receptor context, S1-pocket tyrosines faint. neither is contacted here, so neither
    # picks up the cyan styling below
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "#e8e8e8"}})
    view.addStyle({"chain": "A", "resi": [662, 666]},
                  {"stick": {"color": "#cccccc", "radius": 0.10}})

    # contacted site residues in cyan, catalytic ones marked in their label
    view.addStyle({"chain": "A", "resi": SITE_SHOWN},
                  {"stick": {"colorscheme": "cyanCarbon", "radius": 0.22}})
    for sr in SITE_SHOWN:
        tag = SITE[sr] + (" (catalytic)" if sr in CATALYTIC else "")
        view.addLabel(tag, {"fontSize": FS_SITE, "fontColor": "black", "backgroundColor": "white",
                            "backgroundOpacity": 0.85}, {"chain": "A", "resi": sr, "atom": "CA"})

    # peptide in magenta, with the labelled positions drawn thicker
    view.setStyle({"chain": "B"}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.16}})
    view.addStyle({"chain": "B", "resi": LABEL_PEP},
                  {"stick": {"colorscheme": "magentaCarbon", "radius": 0.27}})
    for pr in LABEL_PEP:
        view.addLabel(PEP[pr], {"fontSize": FS_PEP, "fontColor": "white", "backgroundColor": "#c2185b",
                                "backgroundOpacity": 0.9}, {"chain": "B", "resi": pr, "atom": "CA"})

    # one dashed line per contact, carrying its distance and nudged where two labels collide
    for pr, sr, d, pxyz, sxyz in contacts:
        view.addLine({"dashed": True, "color": "black", "linewidth": LW,
                      "start": {"x": pxyz[0], "y": pxyz[1], "z": pxyz[2]},
                      "end":   {"x": sxyz[0], "y": sxyz[1], "z": sxyz[2]}})
        mid = {"x": (pxyz[0]+sxyz[0])/2, "y": (pxyz[1]+sxyz[1])/2, "z": (pxyz[2]+sxyz[2])/2}
        dx, dy = LABEL_NUDGE.get((pr, sr), (0, 0))
        view.addLabel(f"{d:.1f} A", {"fontSize": FS_DIST, "fontColor": "black", "backgroundColor": "white",
                                     "backgroundOpacity": 0.7, "position": mid,
                                     "screenOffset": {"x": dx*s, "y": dy*s}})

    view.zoomTo({"or": [{"chain": "B", "resi": LABEL_PEP}, {"chain": "A", "resi": SITE_SHOWN}]})
    view.zoom(zoom)
    return view


build_view().show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.